- 如果奖励模型的分数范围负几十到正几十，那么需要对奖励分数做归一化吗，比如0-1之间
    - 你手动将 -50 到 +50 的分数处理成 0 到 1（例如使用 Sigmoid 或 Min-Max Scaling），可能会带来两个严重问题：
        - 梯度消失（Gradient Vanishing）： 如果使用 Sigmoid 函数，当 Reward Model 输出绝对值很大（例如 +50 或 -50）时，Sigmoid 的导数趋近于 0。
        - 分布偏移与不稳定性： 如果使用 Min-Max Scaling（$(x - min)/(max - min)$），你必须在 batch 内或全局统计 min/max。
    - GRPO 算法设计的初衷就是为了解决奖励尺度（Reward Scale）和基线（Baseline）的问题。它不需要外部归一化，因为它的优势函数（Advantage）计算公式自带了“相对化”处理。
        - 在 GRPO 中，对于同一个 Prompt $q$，你会采样 $G$ 个输出 $\{o_1, o_2, ..., o_G\}$。假设 Reward Model 对它们的打分是 $\{r_1, r_2, ..., r_G\}$（哪怕范围是 -50 到 50）。
        - GRPO 计算的优势（Advantage）$A_i$ 公式如下：

$$
A_i = \frac{r_i - \text{mean}(r_{\text{group}})}{\text{std}(r_{\text{group}}) + \epsilon}
$$

- 即：
    - 不要做 (r - min) / (max - min)。
    - 不要做 sigmoid(r)（除非你的 RM 训练时最后就是 Sigmoid，但通常 logits 更好）。
    - 直接使用 RM 的原始 Logits/分数。
    - 依靠 GRPO 算法内部的逻辑（减均值、除方差）来自动适应分数的范围。